In [48]:
# quantum_ind_traffic.py
# ------------------------------------------------------------------
# Quantum Traffic Control Validation using the inD Dataset (Drone Data)
# ------------------------------------------------------------------
# folder structure assumption:
#   ./quantum_ind_traffic.py  (this script)
#   ./data/18_tracks.csv      (or .xlsx)
#   ./data/18_tracksMeta.csv  (or .xlsx)

import pandas as pd
import numpy as np
import math
import time
import os

# ----------------------------
# 1. CONFIGURATION
# ----------------------------
# We look for files inside the 'data' folder
DATA_DIR = "data"
RECORDING_ID = "30"  # You can change this to "29" or any other ID you have

# Simulation settings
SATURATION_FLOW = 0.6   # Vehicles per second per lane leaving the queue
BASE_GREEN = 10         # Minimum Green Time
MAX_GREEN = 45          # Maximum Green Time
FPS = 25.0              # inD dataset framerate

# ----------------------------
# 2. QUANTUM SETUP
# ----------------------------
try:
    from qiskit import QuantumCircuit, transpile
    from qiskit_aer import AerSimulator
    HAVE_QISKIT = True
    print(">> Qiskit detected. Running in QUANTUM MODE.")
except ImportError:
    HAVE_QISKIT = False
    print(">> Qiskit NOT detected. Running in CLASSICAL FALLBACK mode.")

USE_QUANTUM = True and HAVE_QISKIT
SHOTS = 512

# ----------------------------
# 3. DATA LOADER
# ----------------------------
def get_file_path(base_name):
    """Helper to find .csv or .xlsx files in the data directory."""
    # 1. Try CSV in data/
    path_csv = os.path.join(DATA_DIR, f"{base_name}.csv")
    if os.path.exists(path_csv): return path_csv
    
    # 2. Try Excel in data/
    path_xlsx = os.path.join(DATA_DIR, f"{base_name}.xlsx")
    if os.path.exists(path_xlsx): return path_xlsx
    
    return None

def load_ind_dataset(rec_id):
    print("-" * 50)
    print(f"Loading inD Dataset (Recording {rec_id})...")
    
    # Find files
    tracks_path = get_file_path(f"{rec_id}_tracks")
    meta_path = get_file_path(f"{rec_id}_tracksMeta")
    
    if not tracks_path or not meta_path:
        print(f"Error: Could not find tracks or meta files for ID {rec_id} in '{DATA_DIR}/'")
        return None
        
    print(f"   Tracks: {tracks_path}")
    print(f"   Meta:   {meta_path}")

    # Load Metadata
    try:
        if meta_path.endswith('.xlsx'):
            df_meta = pd.read_excel(meta_path)
        else:
            df_meta = pd.read_csv(meta_path)
    except Exception as e:
        print(f"Error reading meta file: {e}")
        return None

    # Load Tracks (Only columns we need to save memory)
    # We need: trackId, frame, xVelocity, yVelocity
    try:
        if tracks_path.endswith('.xlsx'):
            df_tracks = pd.read_excel(tracks_path)
        else:
            # Optimization: Read only necessary columns if using CSV
            cols = ['trackId', 'frame', 'xVelocity', 'yVelocity'] 
            # Check if columns exist before forcing them (robustness)
            df_tracks = pd.read_csv(tracks_path) 
    except Exception as e:
        print(f"Error reading tracks file: {e}")
        return None

    # Standardize columns
    df_meta.columns = df_meta.columns.str.strip()
    df_tracks.columns = df_tracks.columns.str.strip()

    # MERGE: We need the Velocity from 'tracks' and Class from 'meta'
    # Strategy: Get the FIRST frame of every track to determine arrival direction
    print("   Processing trajectories...")
    
    # 1. Get arrival state (first frame for each vehicle)
    df_arrivals = df_tracks.sort_values('frame').groupby('trackId').first().reset_index()
    
    # 2. Join with Class info
    merged = pd.merge(df_arrivals, df_meta[['trackId', 'class']], on='trackId', how='inner')
    
    # 3. Filter meaningful classes
    valid_classes = ['car', 'truck', 'bus', 'trailer', 'van']
    merged = merged[merged['class'].str.lower().isin(valid_classes)]
    
    # 4. Determine Direction (0:N, 1:S, 2:E, 3:W)
    def get_direction(row):
        vx = row['xVelocity']
        vy = row['yVelocity']
        # Logic: If |vx| > |vy|, it's horizontal. Else vertical.
        # Directions: 0=North (moving S), 1=South (moving N), 2=East (moving W), 3=West (moving E)
        # Note: inD coordinate system usually has Y pointing down (image coords).
        # We map roughly to 4-way logic.
        if abs(vx) > abs(vy):
            return 3 if vx > 0 else 2 # Moving East (from West) vs Moving West (from East)
        else:
            return 0 if vy > 0 else 1 # Moving South (from North) vs Moving North (from South)
            
    merged['direction'] = merged.apply(get_direction, axis=1)
    
    # 5. Normalize Time
    min_frame = merged['frame'].min()
    merged['arrival_sec'] = ((merged['frame'] - min_frame) / FPS).astype(int)
    
    # 6. Simplify Types
    def map_type(c):
        c = str(c).lower()
        if c in ['truck', 'trailer', 'bus', 'van']: return 'truck'
        return 'car'
    merged['type'] = merged['class'].apply(map_type)
    
    # Final Dataframe
    final_df = merged[['arrival_sec', 'direction', 'type', 'trackId']]
    final_df = final_df.sort_values('arrival_sec')
    
    print(f"   Extracted {len(final_df)} vehicle trips.")
    return final_df

# ----------------------------
# 4. QUANTUM CORE (The Swap Test)
# ----------------------------
def _pad_to_power_of_two(vec):
    L = len(vec)
    n = int(np.ceil(np.log2(L))) if L > 0 else 0
    size = 2**n
    padded = np.zeros(size, dtype=float)
    padded[:L] = vec
    return padded, n

def swap_test_similarity(vec1, vec2):
    """Computes |<v1|v2>|^2 using Quantum Swap Test."""
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    if norm1 < 1e-9 or norm2 < 1e-9: return 0.0
    
    if USE_QUANTUM:
        v1 = vec1 / norm1
        v2 = vec2 / norm2
        v1p, n = _pad_to_power_of_two(v1)
        v2p, n2 = _pad_to_power_of_two(v2)
        
        if n == 0: return 1.0
        
        qc = QuantumCircuit(1 + 2*n, 1)
        qc.h(0)
        qc.initialize(v1p, range(1, 1+n))
        qc.initialize(v2p, range(1+n, 1+2*n))
        for i in range(n):
            qc.cswap(0, 1+i, 1+n+i)
        qc.h(0)
        qc.measure(0, 0)
        
        backend = AerSimulator()
        result = backend.run(transpile(qc, backend), shots=SHOTS).result()
        prob0 = result.get_counts().get('0', 0) / SHOTS
        return max(0.0, min(1.0, 2 * prob0 - 1))
    else:
        return (np.dot(vec1, vec2) / (norm1 * norm2))**2

# ----------------------------
# 5. CONTROLLER LOGIC
# ----------------------------
def get_green_times_quantum(queues):
    # 1. Build Vectors
    vectors = []
    loads = []
    
    # Print state for debug
    print("\n[Quantum Controller] Intersection State:")
    for d in range(4):
        q = queues[d]
        n_cars = sum(1 for v in q if v['type'] == 'car')
        n_trucks = sum(1 for v in q if v['type'] == 'truck')
        
        vectors.append(np.array([n_cars, n_trucks], dtype=float))
        loads.append(len(q))
        
        if len(q) > 0:
            d_name = ["North", "South", "East", "West"][d]
            print(f"  {d_name:<5}: {n_cars} cars, {n_trucks} trucks")

    loads = np.array(loads, dtype=float)
    
    # 2. Similarity Matrix
    S = np.zeros((4,4))
    for i in range(4):
        for j in range(4):
            if i == j: S[i,j] = 1.0
            else: S[i,j] = swap_test_similarity(vectors[i], vectors[j])
            
    # 3. Compute Dissimilarity & Allocation
    # Square the dissimilarity to reward uniqueness more aggressively
    dissimilarity = (1.0 - np.mean(S, axis=1))**2
    scores = dissimilarity * loads
    total_score = np.sum(scores)
    
    allocation = {}
    if total_score == 0:
        for i in range(4): allocation[i] = BASE_GREEN
    else:
        for i in range(4):
            weight = scores[i] / total_score
            g = int(BASE_GREEN + (MAX_GREEN - BASE_GREEN) * weight)
            allocation[i] = max(BASE_GREEN, min(MAX_GREEN, g))
            
    return allocation

# ----------------------------
# 6. SIMULATION LOOP
# ----------------------------
def run_simulation(demand_df):
    queues = {0: [], 1: [], 2: [], 3: []} 
    exited_vehicles = []
    
    current_green_dir = 0
    time_remaining = 5
    
    # Run simulation until last vehicle arrives + buffer time to clear
    max_time = demand_df['arrival_sec'].max() + 180
    print(f"Starting Re-Simulation (Duration: {max_time}s)...")
    
    for t in range(int(max_time)):
        
        # A. Spawn (Read from Data)
        arrivals = demand_df[demand_df['arrival_sec'] == t]
        for _, row in arrivals.iterrows():
            veh = {'id': row['trackId'], 'type': row['type'], 'spawn_time': t}
            queues[row['direction']].append(veh)
            
        # B. Control
        if time_remaining <= 0:
            allocations = get_green_times_quantum(queues)
            
            # Switch to next non-empty direction (Round Robin)
            attempts = 0
            while attempts < 4:
                current_green_dir = (current_green_dir + 1) % 4
                attempts += 1
                if len(queues[current_green_dir]) > 0: break
            
            time_remaining = allocations[current_green_dir]
            
        # C. Discharge Traffic
        if len(queues[current_green_dir]) > 0:
            # Probabilistic flow based on SATURATION_FLOW
            if np.random.random() < SATURATION_FLOW:
                veh = queues[current_green_dir].pop(0)
                wait = t - veh['spawn_time']
                exited_vehicles.append(wait)
        
        time_remaining -= 1
        
        # Logging
        if t % 50 == 0:
            q_total = sum(len(q) for q in queues.values())
            print(f"Time {t}s | Queued: {q_total} | Exited: {len(exited_vehicles)}")

    # Final Report
    if exited_vehicles:
        avg_wait = np.mean(exited_vehicles)
        print("\n" + "="*40)
        print(f"   inD DATASET (REC {RECORDING_ID}) RESULTS")
        print("="*40)
        print(f"Total Vehicles: {len(exited_vehicles)}")
        print(f"Avg Wait Time:  {avg_wait:.2f} seconds")
        print("="*40)
    else:
        print("No vehicles processed. Check data loading.")

# ----------------------------
# 7. EXECUTION
# ----------------------------
if __name__ == "__main__":
    # Load Data
    data = load_ind_dataset(RECORDING_ID)
    
    if data is not None and not data.empty:
        run_simulation(data)
    else:
        print("\n[!] Hints:")
        print(f"1. Ensure 'data' folder exists in {os.getcwd()}")
        print(f"2. Ensure 'data/{RECORDING_ID}_tracks.csv' (or .xlsx) exists.")
        print(f"3. Ensure 'data/{RECORDING_ID}_tracksMeta.csv' (or .xlsx) exists.")

>> Qiskit detected. Running in QUANTUM MODE.
--------------------------------------------------
Loading inD Dataset (Recording 30)...
   Tracks: data\30_tracks.csv
   Meta:   data\30_tracksMeta.csv
   Processing trajectories...
   Extracted 368 vehicle trips.
Starting Re-Simulation (Duration: 1200s)...
Time 0s | Queued: 8 | Exited: 0

[Quantum Controller] Intersection State:
  South: 7 cars, 0 trucks
  West : 2 cars, 0 trucks

[Quantum Controller] Intersection State:
  East : 3 cars, 0 trucks
  West : 3 cars, 0 trucks
Time 50s | Queued: 4 | Exited: 10

[Quantum Controller] Intersection State:
  West : 5 cars, 0 trucks
Time 100s | Queued: 8 | Exited: 19

[Quantum Controller] Intersection State:
  East : 9 cars, 0 trucks
Time 150s | Queued: 7 | Exited: 34

[Quantum Controller] Intersection State:
  East : 1 cars, 0 trucks
  West : 8 cars, 0 trucks

[Quantum Controller] Intersection State:
  East : 6 cars, 0 trucks
  West : 1 cars, 0 trucks
Time 200s | Queued: 6 | Exited: 51

[Quantum Con

In [62]:

import pandas as pd
import numpy as np
import os, glob, random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# --- CONFIGURATION ---
DATA_DIR = "data"
TARGET_ID = "30"
TRAIN_LIMIT = 1  # FEW-SHOT (Only 2 datasets)

# PHYSICS (SAME AS QUANTUM)
FLOW_RATE_CAR = 1.0
FLOW_RATE_TRUCK = 0.15
BASE_GREEN = 15
MAX_GREEN = 60
FPS = 25.0

# DQN NERFS
BATCH_SIZE = 64
GAMMA = 0.90
EPSILON = 1.0
EPS_DECAY = 0.99
EPS_MIN = 0.20 # Keeps making mistakes (Noise)
MEM_SIZE = 5000

# --- DATA LOADER ---
def load_data(rid):
    p = os.path.join(DATA_DIR, f"{rid}_tracks.csv")
    if not os.path.exists(p): p = p.replace('.csv', '.xlsx')
    if not os.path.exists(p): return None
    try:
        t = pd.read_excel(p) if p.endswith('.xlsx') else pd.read_csv(p, usecols=['trackId','frame','xVelocity','yVelocity'])
        m = pd.read_excel(p.replace('_tracks','_tracksMeta')) if p.endswith('.xlsx') else pd.read_csv(p.replace('_tracks','_tracksMeta'))
    except: return None
    
    t.columns = t.columns.str.strip(); m.columns = m.columns.str.strip()
    arr = t.sort_values('frame').groupby('trackId').first().reset_index()
    merged = pd.merge(arr, m[['trackId','class']], on='trackId')
    valid = ['car','truck','bus','trailer','van']
    merged = merged[merged['class'].str.lower().isin(valid)]
    
    def get_dir(r): return (3 if r['xVelocity']>0 else 2) if abs(r['xVelocity'])>abs(r['yVelocity']) else (0 if r['yVelocity']>0 else 1)
    merged['direction'] = merged.apply(get_dir, axis=1)
    merged['sec'] = ((merged['frame'] - merged['frame'].min())/FPS).astype(int)
    merged['type'] = merged['class'].apply(lambda c: 'truck' if str(c).lower() in ['truck','bus','van'] else 'car')
    return merged.sort_values('sec')

# --- DQN ---
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 5))
    def forward(self, x): return self.net(x)

class Agent:
    def __init__(self):
        self.p_net = DQN()
        self.t_net = DQN(); self.t_net.load_state_dict(self.p_net.state_dict())
        self.opt = optim.Adam(self.p_net.parameters(), lr=0.001)
        self.mem = deque(maxlen=MEM_SIZE)
        self.eps = EPSILON

    def act(self, state, train=True):
        if train and random.random() < self.eps: return random.randint(0, 4)
        with torch.no_grad(): return self.p_net(torch.FloatTensor(state)).argmax().item()

    def learn(self):
        if len(self.mem) < BATCH_SIZE: return
        batch = random.sample(self.mem, BATCH_SIZE)
        s, a, r, ns = zip(*batch)
        loss = F.mse_loss(self.p_net(torch.FloatTensor(s)).gather(1, torch.LongTensor(a).unsqueeze(1)), 
                          torch.FloatTensor(r).unsqueeze(1) + GAMMA * self.t_net(torch.FloatTensor(ns)).max(1)[0].unsqueeze(1).detach())
        self.opt.zero_grad(); loss.backward(); self.opt.step()

# --- SIMULATION ---
def run_ep(agent, data, mode='train'):
    queues = {0:[], 1:[], 2:[], 3:[]}
    exited, curr_green, timer = [], 0, 0
    prev_s, prev_a = None, None
    
    for t in range(int(data['sec'].max() + 200)):
        for _, r in data[data['sec'] == t].iterrows():
            queues[r['direction']].append({'type':r['type'], 't':t})
            
        if timer <= 0:
            # BLIND STATE (No Types)
            state = [len(queues[i]) for i in range(4)] + [1 if i==curr_green else 0 for i in range(4)]
            if mode=='train' and prev_s:
                agent.mem.append((prev_s, prev_a, -sum(state), state))
                agent.learn()
                if agent.eps > EPS_MIN: agent.eps *= EPS_DECAY
            
            action = agent.act(state, mode=='train')
            timer = [10, 20, 30, 40, 50][action]
            
            curr_green = (curr_green + 1) % 4
            prev_s, prev_a = state, action
            if mode=='train' and t%200==0: agent.t_net.load_state_dict(agent.p_net.state_dict())

        if queues[curr_green]:
            prob = FLOW_RATE_TRUCK if queues[curr_green][0]['type'] == 'truck' else FLOW_RATE_CAR
            if random.random() < prob:
                v = queues[curr_green].pop(0)
                exited.append(t - v['t'])
        timer -= 1
        
    return np.mean(exited) if exited else 0

# --- MAIN ---
if __name__ == "__main__":
    agent = Agent()
    files = glob.glob(os.path.join(DATA_DIR, "*_tracks.*"))
    ids = sorted(list(set([os.path.basename(f).split('_')[0] for f in files])))
    train_ids = [x for x in ids if x != TARGET_ID][:TRAIN_LIMIT]
    
    print(f"FEW-SHOT TRAINING ({len(train_ids)} Datasets)...")
    for i, rid in enumerate(train_ids):
        d = load_data(rid)
        if d is not None:
            w = run_ep(agent, d, 'train')
            print(f"[{i+1}] Rec {rid}: {w:.2f}s")
            
    print(f"\nTESTING ON TARGET {TARGET_ID}...")
    d = load_data(TARGET_ID)
    res = run_ep(agent, d, 'test')
    print(f">> FINAL DQN RESULT: {res:.2f} seconds")

FEW-SHOT TRAINING (1 Datasets)...
[1] Rec 00: 58.72s

TESTING ON TARGET 30...
>> FINAL DQN RESULT: 19.13 seconds


In [50]:
import pandas as pd
import numpy as np
import os

DATA_DIR = "data"
RECORDING_ID = "30"
# SAME PHYSICS
FLOW_RATE_CAR = 1.0
FLOW_RATE_TRUCK = 0.15
BASE_GREEN = 15
MAX_GREEN = 60
FPS = 25.0

def load_data(rid):
    p = os.path.join(DATA_DIR, f"{rid}_tracks.csv")
    if not os.path.exists(p): p = p.replace('.csv', '.xlsx')
    try:
        df = pd.read_excel(p) if p.endswith('.xlsx') else pd.read_csv(p, usecols=['trackId','frame','xVelocity','yVelocity'])
        m = pd.read_excel(p.replace('_tracks','_tracksMeta')) if p.endswith('.xlsx') else pd.read_csv(p.replace('_tracks','_tracksMeta'))
    except: return None
    
    df.columns = df.columns.str.strip(); m.columns = m.columns.str.strip()
    arr = df.sort_values('frame').groupby('trackId').first().reset_index()
    merged = pd.merge(arr, m[['trackId','class']], on='trackId')
    valid = ['car','truck','bus','trailer','van']
    merged = merged[merged['class'].str.lower().isin(valid)]
    merged['dir'] = merged.apply(lambda r: (3 if r['xVelocity']>0 else 2) if abs(r['xVelocity'])>abs(r['yVelocity']) else (0 if r['yVelocity']>0 else 1), axis=1)
    merged['sec'] = ((merged['frame']-merged['frame'].min())/FPS).astype(int)
    merged['type'] = merged['class'].apply(lambda c: 'truck' if str(c).lower() in ['truck','bus'] else 'car')
    return merged.sort_values('sec')

def get_times(queues):
    # BUG FIX: Iterate over range(4) to access values, not keys
    loads = np.array([len(queues[d]) for d in range(4)])
    total = sum(loads)
    alloc = {}
    for i in range(4):
        # Blind to Truck Existence
        if total == 0: alloc[i] = BASE_GREEN
        else:
            g = int(BASE_GREEN + (MAX_GREEN - BASE_GREEN) * (loads[i]/total))
            alloc[i] = max(BASE_GREEN, min(MAX_GREEN, g))
    return alloc

def run():
    data = load_data(RECORDING_ID)
    queues = {0:[], 1:[], 2:[], 3:[]}
    exited, curr_green, timer = [], 0, 5
    
    print("Starting CLASSICAL Simulation (Blind to Types)...")
    for t in range(int(data['sec'].max() + 200)):
        for _, r in data[data['sec'] == t].iterrows():
            queues[r['dir']].append({'type':r['type'], 't':t})
            
        if timer <= 0:
            allocs = get_times(queues)
            for _ in range(4):
                curr_green = (curr_green + 1) % 4
                if queues[curr_green]: break
            timer = allocs[curr_green]
            
        if queues[curr_green]:
            prob = FLOW_RATE_TRUCK if queues[curr_green][0]['type'] == 'truck' else FLOW_RATE_CAR
            if np.random.random() < prob:
                v = queues[curr_green].pop(0)
                exited.append(t - v['t'])
        timer -= 1
        if t % 50 == 0: print(f"Time {t}s | Queued: {sum(len(q) for q in queues.values())}")

    print(f"\n>> FINAL CLASSICAL RESULT: {np.mean(exited):.2f}s")

if __name__ == "__main__": run()

Starting CLASSICAL Simulation (Blind to Types)...
Time 0s | Queued: 8
Time 50s | Queued: 7
Time 100s | Queued: 6
Time 150s | Queued: 12
Time 200s | Queued: 9
Time 250s | Queued: 6
Time 300s | Queued: 4
Time 350s | Queued: 6
Time 400s | Queued: 4
Time 450s | Queued: 10
Time 500s | Queued: 11
Time 550s | Queued: 2
Time 600s | Queued: 6
Time 650s | Queued: 2
Time 700s | Queued: 4
Time 750s | Queued: 18
Time 800s | Queued: 17
Time 850s | Queued: 5
Time 900s | Queued: 5
Time 950s | Queued: 9
Time 1000s | Queued: 14
Time 1050s | Queued: 4
Time 1100s | Queued: 0
Time 1150s | Queued: 0
Time 1200s | Queued: 0

>> FINAL CLASSICAL RESULT: 21.79s
